# BHC Human Detection — Faster R-CNN ResNet-50 FPN v2

This notebook trains and evaluates a Faster R-CNN model on the `bhc` dataset stored in Google Drive.

**Dataset:** YOLO-format annotations  
**Classes:** Class 0 = Land, Class 1 = Water  
**Model:** Faster R-CNN ResNet-50 FPN v2  
**Pretrained weights:** COCO  
**Training:** 150 epochs  
**Outputs:** loss graphs, loss components, learning-rate graph, mAP/mAR, precision/recall/F1, confusion matrix, PR curve, inference speed, test predictions, uploaded-image inference, saved model and JSON results.

Run the cells from top to bottom.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("WARNING: GPU is not available.")
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATASET_DRIVE_PATH = "/content/drive/MyDrive/bhc"

print("Dataset path:", DATASET_DRIVE_PATH)
print("Exists:", os.path.exists(DATASET_DRIVE_PATH))

if os.path.exists(DATASET_DRIVE_PATH):
    print("\nFiles/folders:")
    print(os.listdir(DATASET_DRIVE_PATH))
else:
    print("\nERROR: bhc folder was not found.")
    print("Check whether it is directly inside MyDrive.")

In [ ]:
!pip install -q torchmetrics pycocotools albumentations scikit-learn

In [ ]:
import torchmetrics
import sklearn

print("torchmetrics:", torchmetrics.__version__)
print("sklearn:", sklearn.__version__)

In [ ]:
import os

for root, dirs, files in os.walk(DATASET_DRIVE_PATH):
    level = root.replace(DATASET_DRIVE_PATH, '').count(os.sep)

    if level > 3:
        continue

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for f in files[:10]:
        print(f"{indent}  {f}")

In [ ]:
import yaml
import os

DATA_YAML_PATH = os.path.join(DATASET_DRIVE_PATH, "data.yaml")

with open(DATA_YAML_PATH, "r") as f:
    data_yaml = yaml.safe_load(f)

print(data_yaml)

print("\nNumber of classes:", data_yaml["nc"])
print("Class names:", data_yaml["names"])

In [ ]:
CLASS_NAMES_RAW = {
    i: str(name)
    for i, name in enumerate(data_yaml["names"])
}

print("Original dataset classes:")
print(CLASS_NAMES_RAW)

RAW_TO_MODEL = {
    0: 1,
    1: 2
}

MODEL_TO_RAW = {
    1: 0,
    2: 1
}

NUM_CLASSES_WITH_BG = 3

print("\nMapping:")
print("Raw 0 -> Model 1")
print("Raw 1 -> Model 2")
print("Model 0 -> Background")
print("\nTotal model classes:", NUM_CLASSES_WITH_BG)

In [ ]:
from collections import Counter

def count_annotation_classes(split):
    label_dir = os.path.join(DATASET_DRIVE_PATH, split, "labels")
    counter = Counter()
    total_files = 0

    for filename in os.listdir(label_dir):
        if not filename.endswith(".txt"):
            continue

        total_files += 1
        path = os.path.join(label_dir, filename)

        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                cls = int(parts[0])
                counter[cls] += 1

    return total_files, counter

for split in ["train", "valid", "test"]:
    files, counts = count_annotation_classes(split)
    print(f"\n{split.upper()}")
    print("Label files:", files)
    print("Object counts:", dict(counts))

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(DATASET_DRIVE_PATH, split, "images")

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    print(f"{split}: {len(images)} images")

In [ ]:
import shutil
import os

LOCAL_DATASET_PATH = "/content/bhc"

if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)

shutil.copytree(
    DATASET_DRIVE_PATH,
    LOCAL_DATASET_PATH
)

print("Dataset copied to:", LOCAL_DATASET_PATH)

In [ ]:
import os
import time
import random
import numpy as np
import torch

from PIL import Image

from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms.functional as F

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

In [ ]:
class YoloDetectionDataset(Dataset):

    def __init__(self, split_dir):
        self.images_dir = os.path.join(split_dir, "images")
        self.labels_dir = os.path.join(split_dir, "labels")

        self.image_files = sorted([
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]

        img_path = os.path.join(
            self.images_dir,
            img_name
        )

        label_path = os.path.join(
            self.labels_dir,
            os.path.splitext(img_name)[0] + ".txt"
        )

        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        boxes = []
        labels = []

        if os.path.exists(label_path):

            with open(label_path, "r") as f:

                for line in f:

                    parts = line.strip().split()

                    if len(parts) != 5:
                        continue

                    cls, xc, yc, w, h = parts

                    cls = int(cls)
                    xc = float(xc)
                    yc = float(yc)
                    w = float(w)
                    h = float(h)

                    x1 = (xc - w / 2) * img_w
                    y1 = (yc - h / 2) * img_h
                    x2 = (xc + w / 2) * img_w
                    y2 = (yc + h / 2) * img_h

                    x1 = max(0, min(x1, img_w))
                    y1 = max(0, min(y1, img_h))
                    x2 = max(0, min(x2, img_w))
                    y2 = max(0, min(y2, img_h))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(RAW_TO_MODEL[cls])

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        area = (
            (boxes[:, 2] - boxes[:, 0]) *
            (boxes[:, 3] - boxes[:, 1])
        )

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros(
                (len(boxes),),
                dtype=torch.int64
            )
        }

        img = F.to_tensor(img)

        return img, target

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "train")
VALID_DIR = os.path.join(LOCAL_DATASET_PATH, "valid")
TEST_DIR = os.path.join(LOCAL_DATASET_PATH, "test")

train_dataset = YoloDetectionDataset(TRAIN_DIR)
valid_dataset = YoloDetectionDataset(VALID_DIR)
test_dataset = YoloDetectionDataset(TEST_DIR)

print("Train images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Test images:", len(test_dataset))

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

print("DataLoaders created.")

In [ ]:
img, target = train_dataset[0]

print("Image shape:", img.shape)
print("Number of boxes:", len(target["boxes"]))
print("\nBoxes:")
print(target["boxes"])
print("\nModel labels:")
print(target["labels"])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

img, target = train_dataset[0]
img_np = img.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_np)

for box, label in zip(target["boxes"], target["labels"]):

    x1, y1, x2, y2 = box.tolist()

    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    ax.add_patch(rect)

    raw_label = MODEL_TO_RAW[int(label)]

    class_text = "Land" if raw_label == 0 else "Water"

    ax.text(
        x1,
        max(y1 - 5, 5),
        f"{class_text} ({raw_label})",
        color="lime",
        fontsize=10,
        weight="bold"
    )

ax.axis("off")
plt.show()

## IMPORTANT CHECKPOINT

Before starting the 150-epoch training, inspect the image above.

The boxes should be **only around humans**.

If you still see cones, chairs, umbrellas, or other objects, stop and fix the dataset annotations before continuing.

In [ ]:
def get_model():

    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights="DEFAULT"
    )

    in_features = (
        model.roi_heads
        .box_predictor
        .cls_score
        .in_features
    )

    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features,
        NUM_CLASSES_WITH_BG
    )

    return model

model = get_model()
model.to(device)

print("Model created.")

In [ ]:
print(model)

In [ ]:
params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=30,
    gamma=0.1
)

NUM_EPOCHS = 150

print("Epochs:", NUM_EPOCHS)
print("Initial learning rate:", 0.005)
print("LR step size:", 30)
print("LR gamma:", 0.1)

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "loss_classifier": [],
    "loss_box_reg": [],
    "loss_objectness": [],
    "loss_rpn_box_reg": [],
    "learning_rate": []
}

In [ ]:
def evaluate_loss(model, data_loader):

    model.train()

    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():

        for images, targets in data_loader:

            images = [
                img.to(device)
                for img in images
            ]

            targets = [
                {
                    k: v.to(device)
                    for k, v in t.items()
                }
                for t in targets
            ]

            loss_dict = model(images, targets)

            loss = sum(
                loss for loss in loss_dict.values()
            )

            total_loss += loss.item()
            num_batches += 1

    return total_loss / max(num_batches, 1)

In [ ]:
import time
import os
import torch

CHECKPOINT_DIR = os.path.join(
    DATASET_DRIVE_PATH,
    "checkpoints_150_epochs"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    DATASET_DRIVE_PATH,
    "fasterrcnn_bhc_best.pth"
)

FINAL_MODEL_PATH = os.path.join(
    DATASET_DRIVE_PATH,
    "fasterrcnn_bhc_150epochs.pth"
)

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    model.train()

    epoch_loss = 0.0

    epoch_components = {
        "loss_classifier": 0.0,
        "loss_box_reg": 0.0,
        "loss_objectness": 0.0,
        "loss_rpn_box_reg": 0.0
    }

    start_time = time.time()

    for images, targets in train_loader:

        images = [
            img.to(device)
            for img in images
        ]

        targets = [
            {
                k: v.to(device)
                for k, v in t.items()
            }
            for t in targets
        ]

        loss_dict = model(images, targets)

        losses = sum(
            loss for loss in loss_dict.values()
        )

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

        for key in epoch_components:
            if key in loss_dict:
                epoch_components[key] += loss_dict[key].item()

    lr_scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    avg_train_loss = (
        epoch_loss /
        max(len(train_loader), 1)
    )

    avg_val_loss = evaluate_loss(
        model,
        valid_loader
    )

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["learning_rate"].append(current_lr)

    for key in epoch_components:
        history[key].append(
            epoch_components[key] /
            max(len(train_loader), 1)
        )

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        best_text = " ⭐ BEST"

    else:
        best_text = ""

    if (
        (epoch + 1) % 10 == 0
        or epoch == 0
    ):

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"epoch_{epoch+1:03d}.pth"
        )

        torch.save(
            model.state_dict(),
            checkpoint_path
        )

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1:03d}/{NUM_EPOCHS}] "
        f"train={avg_train_loss:.4f} "
        f"val={avg_val_loss:.4f} "
        f"lr={current_lr:.7f} "
        f"time={elapsed:.1f}s"
        f"{best_text}"
    )

torch.save(
    model.state_dict(),
    FINAL_MODEL_PATH
)

print("\nTraining completed.")
print("Final model:", FINAL_MODEL_PATH)
print("Best model:", BEST_MODEL_PATH)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    range(1, NUM_EPOCHS + 1),
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "/content/bhc_loss_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
components = [
    "loss_classifier",
    "loss_box_reg",
    "loss_objectness",
    "loss_rpn_box_reg"
]

plt.figure(figsize=(10, 6))

for key in components:
    plt.plot(
        range(1, NUM_EPOCHS + 1),
        history[key],
        label=key
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Loss Components")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "/content/bhc_loss_components.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    history["learning_rate"]
)

plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule")
plt.grid(alpha=0.3)

plt.savefig(
    "/content/bhc_learning_rate.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
best_model = get_model()

best_model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

best_model.to(device)
best_model.eval()

print("Best model loaded.")
print("Best validation loss:", best_val_loss)

In [ ]:
best_model.eval()

all_preds = []
all_targets = []

start_total = time.time()

with torch.no_grad():

    for images, targets in test_loader:

        images_gpu = [
            img.to(device)
            for img in images
        ]

        outputs = best_model(images_gpu)

        preds = [
            {
                k: v.cpu()
                for k, v in output.items()
            }
            for output in outputs
        ]

        tgts = [
            {
                k: v.cpu()
                for k, v in target.items()
            }
            for target in targets
        ]

        all_preds.extend(preds)
        all_targets.extend(tgts)

total_test_time = time.time() - start_total

print("Test images:", len(all_preds))
print("Total inference time:", round(total_test_time, 2), "seconds")

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(
    class_metrics=True
)

metric.update(
    all_preds,
    all_targets
)

results = metric.compute()

print(
    "mAP@[0.5:0.95]:",
    round(results["map"].item(), 4)
)

print(
    "mAP@0.5:",
    round(results["map_50"].item(), 4)
)

print(
    "mAP@0.75:",
    round(results["map_75"].item(), 4)
)

print(
    "mAR@100:",
    round(results["mar_100"].item(), 4)
)

In [ ]:
if (
    "map_per_class" in results
    and results["map_per_class"].numel() > 0
):

    classes = results.get(
        "classes",
        torch.tensor([1, 2])
    )

    classes = classes.tolist()

    print("\nPer-class AP:")

    for model_label, ap in zip(
        classes,
        results["map_per_class"].tolist()
    ):

        raw_label = MODEL_TO_RAW.get(
            int(model_label),
            int(model_label)
        )

        class_name = "Land" if raw_label == 0 else "Water"

        print(
            f"Class {raw_label} ({class_name}) AP: {ap:.4f}"
        )

In [ ]:
def box_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_width = max(0, x2 - x1)
    inter_height = max(0, y2 - y1)

    intersection = inter_width * inter_height

    area1 = (
        (box1[2] - box1[0]) *
        (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0]) *
        (box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union

In [ ]:
IOU_THRESHOLD = 0.50
SCORE_THRESHOLD = 0.50

y_true = []
y_pred = []
y_scores = []

total_gt = 0
total_matched = 0
total_fp = 0
total_fn = 0

all_detection_records = []

for pred, target in zip(all_preds, all_targets):

    pred_boxes = pred["boxes"].numpy()
    pred_labels = pred["labels"].numpy()
    pred_scores = pred["scores"].numpy()

    gt_boxes = target["boxes"].numpy()
    gt_labels = target["labels"].numpy()

    total_gt += len(gt_boxes)

    keep = pred_scores >= SCORE_THRESHOLD

    pred_boxes = pred_boxes[keep]
    pred_labels = pred_labels[keep]
    pred_scores = pred_scores[keep]

    matched_gt = set()

    order = np.argsort(-pred_scores)

    for pred_idx in order:

        pbox = pred_boxes[pred_idx]
        plabel = pred_labels[pred_idx]
        pscore = pred_scores[pred_idx]

        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):

            if gt_idx in matched_gt:
                continue

            iou = box_iou(pbox, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if (
            best_gt_idx >= 0
            and best_iou >= IOU_THRESHOLD
        ):

            matched_gt.add(best_gt_idx)
            total_matched += 1

            true_label = gt_labels[best_gt_idx]

            y_true.append(int(true_label))
            y_pred.append(int(plabel))
            y_scores.append(float(pscore))

            all_detection_records.append({
                "score": float(pscore),
                "is_tp": True,
                "true_label": int(true_label),
                "pred_label": int(plabel)
            })

        else:

            total_fp += 1

            all_detection_records.append({
                "score": float(pscore),
                "is_tp": False,
                "true_label": None,
                "pred_label": int(plabel)
            })

    image_fn = len(gt_boxes) - len(matched_gt)
    total_fn += image_fn

In [ ]:
detection_precision = (
    total_matched /
    (total_matched + total_fp)
    if (total_matched + total_fp) > 0
    else 0
)

detection_recall = (
    total_matched /
    total_gt
    if total_gt > 0
    else 0
)

detection_f1 = (
    2 *
    detection_precision *
    detection_recall /
    (detection_precision + detection_recall)
    if (detection_precision + detection_recall) > 0
    else 0
)

print("Detection Results")
print("-------------------------")
print("Ground-truth boxes:", total_gt)
print("Matched / TP:", total_matched)
print("False positives:", total_fp)
print("False negatives:", total_fn)
print(f"Detection Precision: {detection_precision:.4f}")
print(f"Detection Recall: {detection_recall:.4f}")
print(f"Detection F1: {detection_f1:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

if len(y_true) > 0:

    classification_accuracy = accuracy_score(
        y_true,
        y_pred
    )

    classification_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    classification_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    classification_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(f"Classification Accuracy: {classification_accuracy:.4f}")
    print(f"Classification Precision: {classification_precision:.4f}")
    print(f"Classification Recall: {classification_recall:.4f}")
    print(f"Classification F1: {classification_f1:.4f}")

else:
    print("No matched detections.")

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        labels=[1, 2],
        target_names=["Land", "Water"],
        zero_division=0
    )
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[1, 2]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Land", "Water"],
    yticklabels=["Land", "Water"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Classification Confusion Matrix")

plt.savefig(
    "/content/bhc_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

records = sorted(
    all_detection_records,
    key=lambda x: x["score"],
    reverse=True
)

if len(records) > 0:

    scores = np.array([
        r["score"] for r in records
    ])

    labels_binary = np.array([
        1 if r["is_tp"] else 0
        for r in records
    ])

    precision, recall, thresholds = precision_recall_curve(
        labels_binary,
        scores
    )

    plt.figure(figsize=(8, 6))

    plt.plot(
        recall,
        precision
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.grid(alpha=0.3)

    plt.savefig(
        "/content/bhc_pr_curve.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
metrics_summary = {
    "mAP@0.5:0.95": float(results["map"].item()),
    "mAP@0.5": float(results["map_50"].item()),
    "mAP@0.75": float(results["map_75"].item()),
    "mAR@100": float(results["mar_100"].item()),
    "Detection Precision": float(detection_precision),
    "Detection Recall": float(detection_recall),
    "Detection F1": float(detection_f1),
    "Classification Accuracy": float(classification_accuracy),
    "Classification Precision": float(classification_precision),
    "Classification Recall": float(classification_recall),
    "Classification F1": float(classification_f1)
}

for name, value in metrics_summary.items():
    print(f"{name:30s}: {value:.4f}")

In [ ]:
metric_names = list(metrics_summary.keys())
metric_values = list(metrics_summary.values())

plt.figure(figsize=(12, 6))

plt.bar(
    metric_names,
    metric_values
)

plt.ylabel("Score")
plt.title("Faster R-CNN Performance Metrics")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    "/content/bhc_metrics_summary.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import time

best_model.eval()
timings = []

with torch.no_grad():

    for i, (images, _) in enumerate(test_loader):

        if i >= 2:
            break

        images_gpu = [
            img.to(device)
            for img in images
        ]

        _ = best_model(images_gpu)

with torch.no_grad():

    for images, _ in test_loader:

        images_gpu = [
            img.to(device)
            for img in images
        ]

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        _ = best_model(images_gpu)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - start

        for _ in images:
            timings.append(
                elapsed / len(images)
            )

avg_inference_time = np.mean(timings)
fps = 1 / avg_inference_time

print(
    f"Average inference time: "
    f"{avg_inference_time * 1000:.2f} ms/image"
)

print(f"FPS: {fps:.2f}")

In [ ]:
def show_test_predictions(
    model,
    dataset,
    num_images=6,
    confidence_threshold=0.60
):

    model.eval()

    num_images = min(
        num_images,
        len(dataset)
    )

    for idx in range(num_images):

        image_tensor, _ = dataset[idx]

        image = (
            image_tensor
            .permute(1, 2, 0)
            .numpy()
        )

        with torch.no_grad():

            prediction = model([
                image_tensor.to(device)
            ])[0]

        boxes = prediction["boxes"].cpu()
        labels = prediction["labels"].cpu()
        scores = prediction["scores"].cpu()

        keep = scores >= confidence_threshold

        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]

        land_count = int((labels == 1).sum())
        water_count = int((labels == 2).sum())
        total_count = land_count + water_count

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(image)

        for box, label, score in zip(
            boxes,
            labels,
            scores
        ):

            raw_label = MODEL_TO_RAW.get(
                int(label),
                int(label)
            )

            x1, y1, x2, y2 = box.numpy()

            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor="lime",
                facecolor="none"
            )

            ax.add_patch(rect)

            class_name = "Land" if raw_label == 0 else "Water"

            ax.text(
                x1,
                max(y1 - 5, 5),
                f"{class_name}:{score:.2f}",
                color="lime",
                fontsize=10,
                weight="bold"
            )

        ax.set_title(
            f"Land: {land_count} | "
            f"Water: {water_count} | "
            f"Total: {total_count}"
        )

        ax.axis("off")
        plt.show()

show_test_predictions(
    best_model,
    test_dataset,
    num_images=6,
    confidence_threshold=0.60
)

In [ ]:
show_test_predictions(
    best_model,
    test_dataset,
    num_images=10,
    confidence_threshold=0.60
)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
CONFIDENCE_THRESHOLD = 0.60

image_name = list(uploaded.keys())[0]

image = Image.open(
    image_name
).convert("RGB")

image_tensor = F.to_tensor(
    image
).to(device)

best_model.eval()

with torch.no_grad():
    prediction = best_model([
        image_tensor
    ])[0]

boxes = prediction["boxes"].cpu()
labels = prediction["labels"].cpu()
scores = prediction["scores"].cpu()

keep = scores >= CONFIDENCE_THRESHOLD

boxes = boxes[keep]
labels = labels[keep]
scores = scores[keep]

land_count = int((labels == 1).sum())
water_count = int((labels == 2).sum())
total_count = land_count + water_count

print("Land:", land_count)
print("Water:", water_count)
print("Total humans:", total_count)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

ax.imshow(image)

for box, label, score in zip(
    boxes,
    labels,
    scores
):

    raw_label = MODEL_TO_RAW.get(
        int(label),
        int(label)
    )

    x1, y1, x2, y2 = box.numpy()

    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    ax.add_patch(rect)

    class_name = "Land" if raw_label == 0 else "Water"

    ax.text(
        x1,
        max(y1 - 5, 5),
        f"{class_name}:{score:.2f}",
        color="lime",
        fontsize=10,
        weight="bold"
    )

ax.set_title(
    f"Human Detection | "
    f"Land: {land_count} | "
    f"Water: {water_count} | "
    f"Total: {total_count}"
)

ax.axis("off")
plt.show()

In [ ]:
import json

final_results = {
    "dataset": "bhc",
    "epochs": NUM_EPOCHS,

    "train_images": len(train_dataset),
    "validation_images": len(valid_dataset),
    "test_images": len(test_dataset),

    "classes": {
        "0": "Land",
        "1": "Water"
    },

    "model": "Faster R-CNN ResNet-50 FPN v2",
    "pretrained_weights": "COCO",

    "metrics": metrics_summary,

    "inference_time_ms": avg_inference_time * 1000,
    "FPS": fps,

    "best_validation_loss": best_val_loss
}

RESULTS_JSON_PATH = os.path.join(
    DATASET_DRIVE_PATH,
    "bhc_final_results.json"
)

with open(RESULTS_JSON_PATH, "w") as f:
    json.dump(
        final_results,
        f,
        indent=4
    )

print("Saved results to:")
print(RESULTS_JSON_PATH)

In [ ]:
import shutil

files_to_save = [
    "/content/bhc_loss_curve.png",
    "/content/bhc_loss_components.png",
    "/content/bhc_learning_rate.png",
    "/content/bhc_confusion_matrix.png",
    "/content/bhc_pr_curve.png",
    "/content/bhc_metrics_summary.png"
]

OUTPUT_DIR = os.path.join(
    DATASET_DRIVE_PATH,
    "results_150_epochs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

for file_path in files_to_save:

    if os.path.exists(file_path):
        shutil.copy(
            file_path,
            OUTPUT_DIR
        )

print("Results saved in:")
print(OUTPUT_DIR)

In [ ]:
print("=" * 60)
print("BHC FASTER R-CNN FINAL RESULTS")
print("=" * 60)

print(f"Training images     : {len(train_dataset)}")
print(f"Validation images   : {len(valid_dataset)}")
print(f"Test images         : {len(test_dataset)}")
print(f"Training epochs     : {NUM_EPOCHS}")

print("\n--- Detection Metrics ---")

print(
    f"mAP@0.5             : "
    f"{metrics_summary['mAP@0.5']:.4f}"
)

print(
    f"mAP@0.5:0.95        : "
    f"{metrics_summary['mAP@0.5:0.95']:.4f}"
)

print(
    f"mAP@0.75            : "
    f"{metrics_summary['mAP@0.75']:.4f}"
)

print(
    f"mAR@100             : "
    f"{metrics_summary['mAR@100']:.4f}"
)

print("\n--- Custom Detection Metrics ---")

print(
    f"Detection Precision : "
    f"{detection_precision:.4f}"
)

print(
    f"Detection Recall    : "
    f"{detection_recall:.4f}"
)

print(
    f"Detection F1        : "
    f"{detection_f1:.4f}"
)

print("\n--- Classification Metrics ---")

print(
    f"Classification Acc  : "
    f"{classification_accuracy:.4f}"
)

print(
    f"Classification Prec : "
    f"{classification_precision:.4f}"
)

print(
    f"Classification Rec  : "
    f"{classification_recall:.4f}"
)

print(
    f"Classification F1   : "
    f"{classification_f1:.4f}"
)

print("\n--- Speed ---")

print(
    f"Inference time      : "
    f"{avg_inference_time * 1000:.2f} ms/image"
)

print(
    f"FPS                 : "
    f"{fps:.2f}"
)

print("=" * 60)

## Final files

The trained models and results are saved in Google Drive:

- `fasterrcnn_bhc_best.pth` — model with lowest validation loss
- `fasterrcnn_bhc_150epochs.pth` — final epoch model
- `bhc_final_results.json` — metrics
- `results_150_epochs/` — graphs
- `checkpoints_150_epochs/` — periodic model checkpoints

### Model summary

**Custom dataset:** BHC, YOLO format  
**Class 0:** Land  
**Class 1:** Water  
**Model:** Faster R-CNN ResNet-50 FPN v2  
**Pretraining:** COCO  
**Fine-tuning:** BHC dataset  
**Epochs:** 150  
**Evaluation:** mAP, mAR, precision, recall, F1, confusion matrix, PR curve, inference speed